# FDCPA Rule Classifier — Dataset Generation

This notebook generates the training dataset using GPT-4.1-mini as a teacher model.

**What it does:**
- Loads the 12-rule FDCPA rubric and 24 hand-curated seed examples
- Uses GPT-4.1-mini to generate ~28 synthetic transcript chunks per rule (easy/medium/hard difficulty)
- Deduplicates near-duplicates using fuzzy matching
- Splits into stratified train/val/test sets

**Estimated API cost:** Free (within OpenAI daily tier limits)

**Prerequisites:** Set `OPENAI_API_KEY` in your environment or `.env` file.

In [ ]:
!pip install openai tqdm python-dotenv

import sys
from pathlib import Path

# Handle both local and Kaggle paths
if Path('../src/fdcpa_classifier').is_dir():
    sys.path.insert(0, '..')
elif Path('/kaggle/working/fdcpa-rule-classifier/src').is_dir():
    sys.path.insert(0, '/kaggle/working/fdcpa-rule-classifier/src')

from dotenv import load_dotenv
load_dotenv()

## 1. Load Rubric and Seed Examples

In [3]:
from fdcpa_classifier import load_rubric
from fdcpa_classifier.prompts import format_teacher_prompt
import json

rubric = load_rubric()
print(f"Loaded {len(rubric)} FDCPA rules")
for r in rubric:
    print(f"  {r['rule_id']}: {r['rule_name']}")

Loaded 12 FDCPA rules
  FDCPA-001: Mini-Miranda Disclosure
  FDCPA-002: Validation of Debt
  FDCPA-003: Call Time Restrictions
  FDCPA-004: Third-Party Disclosure
  FDCPA-005: Harassment or Abuse
  FDCPA-006: False or Misleading Representations
  FDCPA-007: Unfair Practices
  FDCPA-008: Cease and Desist Compliance
  FDCPA-009: Written Notice Requirements
  FDCPA-010: Dispute Handling
  FDCPA-011: Attorney Representation
  FDCPA-012: Threat of Legal Action


In [4]:
with open('../data/seed_examples.json') as f:
    seed_examples = json.load(f)

print(f"Loaded {len(seed_examples)} seed examples")
pass_count = sum(1 for s in seed_examples if s['verdict'] == 'pass')
fail_count = sum(1 for s in seed_examples if s['verdict'] == 'fail')
print(f"  Pass: {pass_count}, Fail: {fail_count}")

Loaded 24 seed examples
  Pass: 12, Fail: 12


## 2. Preview Teacher Prompt

In [5]:
rule_001 = rubric[0]
rule_001_seeds = [s for s in seed_examples if s['rule_id'] == 'FDCPA-001']
prompt = format_teacher_prompt(rule_001, rule_001_seeds)
print(prompt[:2000])
print(f"\n... (total prompt length: {len(prompt)} chars)")

## Task
Generate 5 transcript chunks for the following FDCPA rule. Each chunk should be 200–400 words of realistic collections call dialog.

## Rule
- **Rule ID:** FDCPA-001
- **Rule Name:** Mini-Miranda Disclosure
- **Description:** The debt collector must identify themselves as a debt collector and state that any information obtained will be used for that purpose within the first communication or within 5 days of the initial communication.
- **Legal Basis:** 15 U.S.C. § 1692e(11)

## Difficulty Distribution
Generate chunks at these difficulty levels:
- 2 easy (clear pass or clear fail based on obvious compliance/violation)
- 2 medium (requires careful analysis, borderline cases)
- 1 hard (ambiguous, expert-level judgment needed)

## Seed Examples (for style reference)
### Seed Example 1
Verdict: pass
Transcript:
Agent: Good morning, this is Alex from [AGENCY_NAME]. I'm calling in regards to an account for [CONSUMER_NAME]. Before we proceed, I need to let you know that I am a debt col

## 3. Smoke Test: Generate for FDCPA-001

In [6]:
from fdcpa_classifier.data_gen import generate_examples_for_rule

test_examples = generate_examples_for_rule(
    rule_001,
    n_easy=3,
    n_medium=2,
    n_hard=1,
    seed_examples=seed_examples,
)
print(f"Generated {len(test_examples)} examples for {rule_001['rule_id']}")
for ex in test_examples[:2]:
    print(f"\n  Verdict: {ex['verdict']}, Difficulty: {ex.get('difficulty', 'N/A')}")
    print(f"  Transcript preview: {ex['transcript_chunk'][:150]}...")

Generated 5 examples for FDCPA-001

  Verdict: pass, Difficulty: easy
  Transcript preview: Agent: Hello, this is Maria calling from [AGENCY_NAME]. I want to inform you that I am a debt collector, and any information obtained will be used for...

  Verdict: fail, Difficulty: easy
  Transcript preview: Agent: Good afternoon, may I speak with [CONSUMER_NAME], please?
Consumer: Speaking. Who is this?
Agent: This is Tom from [AGENCY_NAME]. We are contac...


## 4. Full Generation Across All 12 Rules

In [8]:
from fdcpa_classifier.data_gen import generate_dataset

train, val, test = generate_dataset(
    rubric=rubric,
    n_per_rule_easy=15,
    n_per_rule_medium=8,
    n_per_rule_hard=5,
)

Generating examples per rule:   0%|                             | 0/12 [00:00<?, ?it/s]


Processing FDCPA-001: Mini-Miranda Disclosure


Generating examples per rule:   8%|█▊                   | 1/12 [01:08<12:34, 68.59s/it]

  Generated 23 new examples (total: 23)

Processing FDCPA-002: Validation of Debt


Generating examples per rule:  17%|███▌                 | 2/12 [02:16<11:21, 68.15s/it]

...

Generating examples per rule:  92%|██████████████████▎ | 11/12 [12:45<01:10, 70.46s/it]

  Generated 25 new examples (total: 247)

Processing FDCPA-012: Threat of Legal Action


Generating examples per rule: 100%|████████████████████| 12/12 [13:52<00:00, 69.37s/it]

  Generated 25 new examples (total: 271)

Total before dedup: 271


After dedup: 271

Dataset saved:
  Train: 208
  Val:   24
  Test:  39


## 5. Dataset Statistics

In [9]:
import pandas as pd

all_data = train + val + test
df = pd.DataFrame(all_data)

print(f"\nTotal examples: {len(df)}")
print(f"\nSplit sizes: Train={len(train)}, Val={len(val)}, Test={len(test)}")
print(f"\nVerdict distribution:")
print(df['verdict'].value_counts())
print(f"\nExamples per rule:")
print(df.groupby('rule_id').size().sort_index())
print(f"\nDifficulty distribution:")
print(df['difficulty'].value_counts())


Total examples: 271

Split sizes: Train=208, Val=24, Test=39

Verdict distribution:
verdict
pass    152
fail    119
Name: count, dtype: int64

Examples per rule:
rule_id
FDCPA-001    23
FDCPA-002    23
FDCPA-003    23
FDCPA-004    22
FDCPA-005    23
FDCPA-006    23
FDCPA-007    21
FDCPA-008    23
FDCPA-009    20
FDCPA-010    21
FDCPA-011    25
FDCPA-012    24
dtype: int64

Difficulty distribution:
difficulty
easy      113
medium    112
hard       46
Name: count, dtype: int64


## 6. Sanity Check: Display Examples from Each Split

In [10]:
for split_name, split_data in [('Train', train), ('Val', val), ('Test', test)]:
    print(f"\n{'='*60}")
    print(f"{split_name} split — example:")
    print(f"{'='*60}")
    if split_data:
        ex = split_data[0]
        print(f"Rule: {ex['rule_id']}")
        print(f"Verdict: {ex['verdict']}")
        print(f"Difficulty: {ex.get('difficulty', 'N/A')}")
        print(f"Transcript:\n{ex['transcript_chunk'][:400]}...")
        print(f"Reasoning: {ex.get('reasoning', 'N/A')}")


Train split — example:
Rule: FDCPA-002
Verdict: fail
Difficulty: medium
Transcript:
Consumer: I received your letter about a debt. I want to request validation in writing.
Agent: Sure, just send us a letter and we’ll get back to you.
Consumer: Can you explain what you'll send? I want to be sure it’s thorough.
Agent: We generally send the balance and the creditor name.
Consumer: But do you send it automatically when the request is received?
Agent: Usually, yes. But sometimes it ca...
Reasoning: Agent is vague about suspension of collection activities and concedes calls may continue despite validation requests which violates the requirement to pause collection until verification is provided.

Val split — example:
Rule: FDCPA-012
Verdict: fail
Difficulty: medium
Transcript:
Agent: Good afternoon, this is Lisa from [AGENCY_NAME]. Your balance of $[AMOUNT] on account [ACCOUNT_NUMBER] is seriously overdue. If payment isn’t made, we may have your wages garnished by next month.
Consumer: Do y

## Next Steps

1. **Hand-review the test set** (`data/test.jsonl`) — verify each example's verdict and reasoning
2. Fix any incorrect labels and add `"human_reviewed": true` to each test example
3. Document review notes in `data/REVIEW_NOTES.md`
4. Proceed to `02_train_qlora.ipynb` for training